# Kaggle: consultant servers + Track 1 on seed_30

Run top to bottom on **Kaggle, 2×T4, internet on**. Add `HF_TOKEN` in *Add-ons → Secrets* (the AWQ repos are
ungated, so it is optional), and `GITHUB_TOKEN` too if the repo is private.

What it does:
1. Clones the repo, makes a venv, installs the pinned vLLM and our package.
2. Starts one vLLM server per T4 (Qwen3-8B-AWQ on GPU 0 / :8000, Llama-3.1-8B AWQ on GPU 1 / :8001) with the flags
   and pinned revisions from `configs/models.yaml`, and waits for `/health`.
3. Writes `.env` with `VLLM_BASE_URL_A/B`.
4. Smoke test through **our backend** (`crux smoke consultants`): same per-request settings as the real stages,
   including Qwen's `enable_thinking: false`, but with `<think>` stripping off, so it fails if thinking is on, if a
   reply is truncated, if the JSON doesn't parse, or if the wrong model answers. **Everything stops here on failure.**
5. Runs `crux track1 propose | leakage --l2 | decisive`, then `crux track1 report`.
6. Zips `data/runs/<RUN_NAME>/`, the server logs and a run manifest into `/kaggle/working/<RUN_NAME>.zip`.

All settings are in the next cell. Model flags are not repeated here: they are read from `configs/models.yaml`.

## 1. Settings

In [ ]:
# ---- Every setting for this run lives in this cell ----
REPO_URL = "https://github.com/swaroop05v/Crux-Audited-Consensus.git"
REPO_BRANCH = "main"
REPO_DIR = "/kaggle/working/capstone"
VENV_DIR = "/kaggle/working/venv"
LOG_DIR = "/kaggle/working/logs"

VLLM_VERSION = "0.32.0"    # pinned: the version of Sruthi's smoke test (30 Oct); keep it for the whole kill test

RUN_NAME = "seed30_v2"     # stage files go to data/runs/<RUN_NAME>/ (seed30_v1: first run, prompt v2)
PROPOSAL_PROMPT_VERSION = "v3"  # Prompt 1A version: v2 = as in seed30_v1; v3 = asks for a two-way crux
CASES = "data/seed/seed_30.jsonl"  # relative to REPO_DIR
SEED = 42
K = 3                      # decisiveness samples per consultant (Doc 2)
RUN_L2 = True              # restatement probe (Prompt 1C) on both consultants (decisions D1, P4)
SMOKE_N = 2                # seed cases in the smoke test

# Serving: one T4 per consultant (decisions D9). Model flags come from configs/models.yaml.
GPUS = {"consultant_a": "0", "consultant_b": "1"}
PORTS = {"consultant_a": 8000, "consultant_b": 8001}
MAX_NUM_SEQS = 8           # concurrent sequences per server (not 1: that serializes everything)
WORKERS = 2 * MAX_NUM_SEQS  # concurrent requests from the crux CLI, split across the two servers
GPU_MEMORY_UTILIZATION = 0.90
# Use only the sampling parameters we send (decisions D3), not defaults from the model repo's
# generation_config.json (Qwen3's sets top_k=20). Drop this if the pinned vLLM rejects the flag.
VLLM_EXTRA_ARGS = ["--generation-config", "vllm"]
HEALTH_TIMEOUT_S = 1200    # model download + load can take several minutes per server

In [ ]:
import json
import os
import shutil
import signal
import subprocess
import sys
import time
import urllib.request


def get_secret(name):
    """Kaggle secret, else environment variable, else None."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)


HF_TOKEN = get_secret("HF_TOKEN")
GITHUB_TOKEN = get_secret("GITHUB_TOKEN")  # only needed if the repo is private
print("HF_TOKEN:", "set" if HF_TOKEN else "not set (fine: the AWQ repos are ungated)")
print("GITHUB_TOKEN:", "set" if GITHUB_TOKEN else "not set")


def sh(cmd, cwd=None, env=None, shown=None):
    """Run a command, streaming its output; raise if it fails. ``shown`` replaces the echoed command
    (used to keep tokens out of the log)."""
    print("$", shown or " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {shown or cmd[:3]}")


def tail(path, lines=60):
    with open(path, encoding="utf-8", errors="replace") as handle:
        return "".join(handle.readlines()[-lines:])

## 2. Clone and install

In [ ]:
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
clone_url = (REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@") if GITHUB_TOKEN
             else REPO_URL)
sh(["git", "clone", "--branch", REPO_BRANCH, clone_url, REPO_DIR],
   shown=f"git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
sh(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_DIR)  # keep the token out of .git/config
# Our submodule only; its own nested llm_debate submodule is not needed for this run.
sh(["git", "submodule", "update", "--init", "collaborative-dr"], cwd=REPO_DIR)
REPO_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("repo commit:", REPO_COMMIT)

In [ ]:
# Kaggle's system Python has no ensurepip, so `python -m venv` cannot install pip into a venv.
# virtualenv bundles its own pip. Remove any half-built venv from an earlier failed attempt first.
shutil.rmtree(VENV_DIR, ignore_errors=True)
sh([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])
sh([sys.executable, "-m", "virtualenv", "-q", VENV_DIR])
PY = f"{VENV_DIR}/bin/python"
sh([PY, "-m", "pip", "install", "-q", "--upgrade", "pip"])
sh([PY, "-m", "pip", "install", "-q", f"vllm=={VLLM_VERSION}"])
sh([PY, "-m", "pip", "install", "-q", "-e", REPO_DIR])
sh([PY, "-c", "import sys, torch, vllm; print('python', sys.version.split()[0]); print('vllm', vllm.__version__); "
              "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| gpus', torch.cuda.device_count())"])
sh(["nvidia-smi"])

## 3. Start one vLLM server per T4

In [ ]:
import os

import yaml

with open(os.path.join(REPO_DIR, "configs", "models.yaml"), encoding="utf-8") as handle:
    MODELS = yaml.safe_load(handle)["models"]


def vllm_command(model_key):
    """vllm serve command for one consultant, from its pinned entry in configs/models.yaml."""
    spec = MODELS[model_key]
    assert spec["backend"] == "vllm", f"{model_key} is not served by vLLM"
    return [
        f"{VENV_DIR}/bin/vllm", "serve", spec["hf_repo"],
        "--revision", spec["revision"],
        "--tokenizer-revision", spec["revision"],
        "--served-model-name", spec["served_name"],
        "--quantization", spec["quantization"],
        "--dtype", spec["dtype"],
        "--max-model-len", str(spec["max_model_len"]),
        "--tensor-parallel-size", str(spec["tensor_parallel_size"]),
        "--max-num-seqs", str(MAX_NUM_SEQS),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--host", "127.0.0.1",
        "--port", str(PORTS[model_key]),
        "--seed", str(SEED),
        *VLLM_EXTRA_ARGS,
    ]


SERVER_CMDS = {key: vllm_command(key) for key in PORTS}
for key, cmd in SERVER_CMDS.items():
    print(f"{key} (GPU {GPUS[key]}):", " ".join(cmd))

In [ ]:
os.makedirs(LOG_DIR, exist_ok=True)
SERVERS = {}
for key, cmd in SERVER_CMDS.items():
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": GPUS[key]}
    if HF_TOKEN:
        env["HF_TOKEN"] = HF_TOKEN
    log_path = f"{LOG_DIR}/vllm_{key}.log"
    log = open(log_path, "w")
    SERVERS[key] = {"proc": subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env,
                                             start_new_session=True),
                    "log": log, "log_path": log_path}


def get_json(url):
    with urllib.request.urlopen(url, timeout=5) as response:
        return response.status, json.loads(response.read() or b"null")


def healthy(port):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=5) as response:
            return response.status == 200
    except Exception:
        return False


start, pending = time.time(), set(SERVERS)
while pending and time.time() - start < HEALTH_TIMEOUT_S:
    for key in sorted(pending):
        if SERVERS[key]["proc"].poll() is not None:
            raise RuntimeError(f"{key} server exited (code {SERVERS[key]['proc'].returncode}):\n"
                               + tail(SERVERS[key]["log_path"]))
        if healthy(PORTS[key]):
            print(f"{key} healthy after {time.time() - start:.0f}s")
            pending.discard(key)
    if pending:
        time.sleep(10)
if pending:
    for key in pending:
        print(f"===== {key} log tail =====\n" + tail(SERVERS[key]["log_path"]))
    raise RuntimeError(f"not healthy after {HEALTH_TIMEOUT_S}s: {sorted(pending)}")

for key in SERVERS:  # the right model behind each port
    _, models = get_json(f"http://127.0.0.1:{PORTS[key]}/v1/models")
    served = [m["id"] for m in models["data"]]
    assert MODELS[key]["served_name"] in served, f"{key}: port {PORTS[key]} serves {served}"
    print(f"{key}: {served}")

In [ ]:
# Our backend reads the per-model URL variables named in configs/models.yaml (base_url_env).
with open(os.path.join(REPO_DIR, ".env"), "w") as handle:
    for key in PORTS:
        handle.write(f"{MODELS[key]['base_url_env']}=http://127.0.0.1:{PORTS[key]}/v1\n")
print(open(os.path.join(REPO_DIR, ".env")).read())

## 4. Smoke test through our backend (stops the notebook on failure)

In [ ]:
# `crux` commands this notebook runs (unit-tested against the CLI in tests/test_kaggle_notebook.py).
SMOKE_CMD = ["smoke", "consultants", "--backend", "vllm", "--cases", CASES, "--n", str(SMOKE_N),
             "--seed", str(SEED)]


def crux_steps():
    common = ["--run", RUN_NAME, "--cases", CASES, "--backend", "vllm", "--seed", str(SEED),
              "--workers", str(WORKERS)]
    return [
        ["track1", "propose", *common, "--prompt-version", PROPOSAL_PROMPT_VERSION],
        ["track1", "leakage", *common, *(["--l2"] if RUN_L2 else [])],
        ["track1", "decisive", *common, "--k", str(K)],
        ["track1", "report", "--run", RUN_NAME],
    ]

In [ ]:
STEP_SECONDS = {}


def run_crux(args):
    started = time.time()
    sh([PY, "-m", "crux", *args], cwd=REPO_DIR, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    STEP_SECONDS[" ".join(args[:2])] = round(time.time() - started, 1)
    print(f"--- {' '.join(args[:2])}: {STEP_SECONDS[' '.join(args[:2])]}s")


run_crux(SMOKE_CMD)  # raises on any failed check: thinking on, truncated, bad JSON, wrong model

## 5. Track 1 on seed_30

In [ ]:
for step in crux_steps():
    run_crux(step)

## 6. Package the outputs

In [ ]:
run_dir = os.path.join(REPO_DIR, "data", "runs", RUN_NAME)
for key, server in SERVERS.items():
    server["log"].flush()
    shutil.copy(server["log_path"], os.path.join(run_dir, f"vllm_{key}.log"))
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout
vllm_version = subprocess.check_output([PY, "-c", "import vllm; print(vllm.__version__)"], text=True).strip()
manifest = {
    "run_name": RUN_NAME,
    "repo_url": REPO_URL,
    "repo_branch": REPO_BRANCH,
    "repo_commit": REPO_COMMIT,
    "vllm_version": vllm_version,
    "settings": {name: globals()[name] for name in (
        "VLLM_VERSION", "RUN_NAME", "PROPOSAL_PROMPT_VERSION", "CASES", "SEED", "K", "RUN_L2", "SMOKE_N", "GPUS", "PORTS", "MAX_NUM_SEQS",
        "WORKERS", "GPU_MEMORY_UTILIZATION", "VLLM_EXTRA_ARGS")},
    "server_commands": SERVER_CMDS,
    "step_seconds": STEP_SECONDS,
    "gpus": gpu,
    "finished_at": time.strftime("%Y-%m-%d %H:%M:%S %Z"),
}
with open(os.path.join(run_dir, "kaggle_manifest.json"), "w") as handle:
    json.dump(manifest, handle, indent=2)
archive = shutil.make_archive(f"/kaggle/working/{RUN_NAME}", "zip", root_dir=os.path.dirname(run_dir),
                              base_dir=RUN_NAME)
print("download:", archive)

In [ ]:
for key, server in SERVERS.items():  # stop the servers
    try:
        os.killpg(server["proc"].pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    server["log"].close()
print("servers stopped")